In [1]:
import re
import warnings
import urllib.request

import nltk
import optuna
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from corus import load_lenta

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
SEED = 42

d:\vscode_projects\itmo_dl_nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Загрузка, предобработка

In [2]:
# Используем urllib для скачивания данных (если файл отсутствует)
data_url = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
local_filename = "lenta-ru-news.csv.gz"

try:
    with open(local_filename, "rb"):
        pass
except FileNotFoundError:
    print("Downloading dataset...")
    urllib.request.urlretrieve(data_url, local_filename)
    print("Download completed.")

In [3]:
# Загружаем данные с помощью Corus
raw_records = load_lenta(local_filename)
df = pd.DataFrame(raw_records)
df.columns = ['url', 'title', 'text', 'topic', 'tags', 'date']
df = df[['title', 'text', 'topic']]
df = df.sample(n=100_000, random_state=SEED).reset_index(drop=True)
df.head()

,title,text,topic
0,EgyptAir объявила о подорожании билетов,Египетский перевозчик EgyptAir сообщил о возмо...,Путешествия
1,Глава Красногорского района Подмосковья ушел в...,Глава Красногорского района Московской области...,Россия
2,Милонов предложил запретить россиянам сидеть в...,Депутат Виталий Милонов внес в Госдуму законоп...,Россия
3,Женщинам в детородном возрасте разрешили посещ...,Верховный суд Индии разрешил женщинам в фертил...,Мир
4,Россиянам пообещали дешевый хлеб,Россиянам не стоит бояться роста цен на хлеб —...,Экономика


In [4]:
# Фильтрация редких классов: оставляем только топики с не менее чем 1 примерами
topic_freq = df['topic'].value_counts()
popular_topics = topic_freq[topic_freq > 1].index
df = df[df['topic'].isin(popular_topics)].reset_index(drop=True)
df['topic'].value_counts()

topic
Россия               21871
Мир                  18494
Экономика            10737
Спорт                 8632
Культура              7337
Наука и техника       7129
Бывший СССР           7100
Интернет и СМИ        6181
Из жизни              3718
Дом                   2891
Силовые структуры     2661
Ценности              1079
Бизнес                 967
Путешествия            855
69-я параллель         178
Крым                    82
Культпросвет            45
                        23
Легпром                 10
Библиотека               8
Name: count, dtype: int64

In [5]:
# Загрузка инструментов для обработки текста
nltk.download('stopwords')
russian_stop = set(stopwords.words("russian"))
stemmer = SnowballStemmer("russian")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ivann\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
def clean_text(raw_text: str) -> str:
    """
    Функция очистки текста с использованием стемминга:
    - Приведение к нижнему регистру,
    - Удаление HTML-тегов,
    - Сохранение русских слов и удаление пунктуации,
    - Токенизация, удаление стоп-слов и стемминг.
    """
    txt = raw_text.lower()
    txt = re.sub(r"<.*?>", " ", txt)
    txt = re.sub(r"[^а-яё\s]", " ", txt)
    tokens = [w for w in txt.split() if w not in russian_stop]
    stemmed = [stemmer.stem(token) for token in tokens]
    return " ".join(stemmed)

Базовая предобработка, включающая очистку (удаление HTML-тегов (не проверял, но такое может быть), перевод в нижний регистр, фильтрация символов и удаление стоп-слов) и стемминг, была выбрана для быстрой обработки данных и уменьшения вычислительных затрат. Базовая обработка для удаления шума из данных и снижения размерности признакового пространства. Сохраняю только русские слова для снижения размера признаков (наверняка слова английские встречаются слишком редко).

In [7]:
df["full_text"] = (df["title"] + " " + df["text"]).apply(clean_text)
df["full_text"].head()

0    объяв подорожан билет египетск перевозчик сооб...
1    глав красногорск район подмосков ушел отставк ...
2    милон предлож запрет россиян сидет соцсет рабо...
3    женщин детородн возраст разреш посеща индуистс...
4    россиян пообеща дешев хлеб россиян сто боя рос...
Name: full_text, dtype: object

In [8]:
# Разбиваем данные на train/validation/test (60/20/20) с сохранением пропорций классов
X = df["full_text"]
y = df["topic"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size=0.6, stratify=y, random_state=SEED)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

In [9]:
# --- Dummy baseline ---
dummy_pipe = Pipeline([
    ("vec", CountVectorizer()),
    ("dummy", DummyClassifier(strategy="most_frequent", random_state=SEED))
])
dummy_pipe.fit(X_train, y_train)
dummy_preds = dummy_pipe.predict(X_valid)
print("Dummy baseline report:")
print(classification_report(y_valid, dummy_preds))

Dummy baseline report:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       0.00      0.00      0.00        36
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.00      0.00      0.00       193
      Бывший СССР       0.00      0.00      0.00      1420
              Дом       0.00      0.00      0.00       578
         Из жизни       0.00      0.00      0.00       743
   Интернет и СМИ       0.00      0.00      0.00      1236
             Крым       0.00      0.00      0.00        16
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.00      0.00      0.00      1468
          Легпром       0.00      0.00      0.00         2
              Мир       0.00      0.00      0.00      3699
  Наука и техника       0.00      0.00      0.00      1426
      Путешествия       0.00      0.00      0.00       171
           Россия       0.22    

#### Пайплайны с LogisticRegression
Ниже приведены два варианта: с использованием CountVectorizer и TfidfVectorizer.
Сначала базовые модели без оптимизации.

In [10]:
# Базовая модель с CountVectorizer
pipe_count = Pipeline([
    ("vec", CountVectorizer()),
    ("clf", LogisticRegression(random_state=SEED, max_iter=1000))
])
pipe_count.fit(X_train, y_train)
count_preds = pipe_count.predict(X_valid)
print("Report (CountVectorizer + LogReg):")
print(classification_report(y_valid, count_preds))

Report (CountVectorizer + LogReg):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       0.77      0.47      0.59        36
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.51      0.37      0.43       193
      Бывший СССР       0.82      0.79      0.80      1420
              Дом       0.86      0.77      0.81       578
         Из жизни       0.61      0.55      0.57       743
   Интернет и СМИ       0.73      0.71      0.72      1236
             Крым       0.83      0.31      0.45        16
    Культпросвет        0.50      0.11      0.18         9
         Культура       0.87      0.86      0.87      1468
          Легпром       0.00      0.00      0.00         2
              Мир       0.78      0.80      0.79      3699
  Наука и техника       0.79      0.79      0.79      1426
      Путешествия       0.80      0.63      0.70       171
           Россия   

In [11]:
# Базовая модель с TfidfVectorizer
pipe_tfidf = Pipeline([
    ("vec", TfidfVectorizer()),
    ("clf", LogisticRegression(random_state=SEED, max_iter=1000))
])
pipe_tfidf.fit(X_train, y_train)
tfidf_preds = pipe_tfidf.predict(X_valid)
print("Report (TfidfVectorizer + LogReg):")
print(classification_report(y_valid, tfidf_preds))

Report (TfidfVectorizer + LogReg):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       1.00      0.03      0.05        36
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.84      0.16      0.27       193
      Бывший СССР       0.82      0.79      0.81      1420
              Дом       0.87      0.72      0.79       578
         Из жизни       0.68      0.53      0.59       743
   Интернет и СМИ       0.78      0.69      0.73      1236
             Крым       0.00      0.00      0.00        16
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.85      0.89      0.87      1468
          Легпром       0.00      0.00      0.00         2
              Мир       0.78      0.85      0.81      3699
  Наука и техника       0.81      0.82      0.82      1426
      Путешествия       0.88      0.51      0.64       171
           Россия   

#### Оптимизация гиперпараметров с использованием Optuna

Здесь подбираю одновременно параметры векторизаторов и LogisticRegression. Для каждого варианта (CountVectorizer и TfidfVectorizer) определим отдельную функцию-цель.

In [12]:
def objective_count(trial: optuna.Trial) -> float:
    # Параметры векторизатора
    max_df = trial.suggest_float("max_df", 0.7, 0.9)
    min_df = trial.suggest_float("min_df", 0.003, 0.01)
    ngram_low = 1
    ngram_high = trial.suggest_int("ngram_high", 1, 2)
    
    # Параметры классификатора
    C_val = trial.suggest_float("C", 0.02, 5.0, log=True)
    solver_opt = trial.suggest_categorical("solver", ["liblinear", "lbfgs"])
    if solver_opt == "liblinear":
        penalty_opt = trial.suggest_categorical("penalty", ["l1", "l2"])
    else:
        penalty_opt = "l2"
    
    pipe = Pipeline([
        ("vec", CountVectorizer(max_df=max_df, min_df=min_df, ngram_range=(ngram_low, ngram_high))),
        ("clf", LogisticRegression(C=C_val, solver=solver_opt, penalty=penalty_opt,
                                     random_state=SEED, max_iter=1000))
    ])
    
    # Используем кросс-валидацию для оценки 
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy", n_jobs=-1)
    return scores.mean()

In [13]:
study_count = optuna.create_study(direction="maximize")
study_count.optimize(objective_count, n_trials=20)

print("Best parameters for CountVectorizer pipeline:")
print(study_count.best_params)
print("Best CV accuracy:", study_count.best_value)

[I 2025-03-13 19:19:32,664] A new study created in memory with name: no-name-b046e161-5884-45cd-ba73-6a597b13813f
[I 2025-03-13 19:20:33,091] Trial 0 finished with value: 0.7150237464233131 and parameters: {'max_df': 0.8943162100877238, 'min_df': 0.007100025867265267, 'ngram_high': 2, 'C': 3.773320335467855, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.7150237464233131.
[I 2025-03-13 19:23:59,369] Trial 1 finished with value: 0.7494915256826957 and parameters: {'max_df': 0.8836580209435794, 'min_df': 0.008751429341076266, 'ngram_high': 2, 'C': 3.259567464020644, 'solver': 'liblinear', 'penalty': 'l2'}. Best is trial 1 with value: 0.7494915256826957.
[I 2025-03-13 19:24:32,400] Trial 2 finished with value: 0.7556250798677668 and parameters: {'max_df': 0.8012496871245457, 'min_df': 0.006473252973126354, 'ngram_high': 1, 'C': 1.9131369478135847, 'solver': 'liblinear', 'penalty': 'l1'}. Best is trial 2 with value: 0.7556250798677668.
[I 2025-03-13 19:26:29,067] Trial 3 finished with v

Best parameters for CountVectorizer pipeline:
{'max_df': 0.84497947612502, 'min_df': 0.0031232550860989814, 'ngram_high': 1, 'C': 0.02159705836790944, 'solver': 'lbfgs'}
Best CV accuracy: 0.7887928160680057


In [14]:
def objective_tfidf(trial: optuna.Trial) -> float:
    # Параметры векторизатора TF-IDF
    max_df = trial.suggest_float("max_df", 0.7, 0.9)
    min_df = trial.suggest_float("min_df", 0.003, 0.01)
    ngram_low = 1
    ngram_high = trial.suggest_int("ngram_high", 1, 2)
    
    # Параметры классификатора
    C_val = trial.suggest_float("C", 0.02, 5.0, log=True)
    solver_opt = trial.suggest_categorical("solver", ["liblinear", "lbfgs"])
    if solver_opt == "liblinear":
        penalty_opt = trial.suggest_categorical("penalty", ["l1", "l2"])
    else:
        penalty_opt = "l2"
    
    pipe = Pipeline([
        ("vec", TfidfVectorizer(max_df=max_df, min_df=min_df, ngram_range=(ngram_low, ngram_high))),
        ("clf", LogisticRegression(C=C_val, solver=solver_opt, penalty=penalty_opt,
                                     random_state=SEED, max_iter=1000))
    ])
    
    scores = cross_val_score(pipe, X_train, y_train, cv=3, scoring="accuracy", n_jobs=-1)
    return scores.mean()

In [15]:
study_tfidf = optuna.create_study(direction="maximize")
study_tfidf.optimize(objective_tfidf, n_trials=20)

print("Best parameters for TfidfVectorizer pipeline:")
print(study_tfidf.best_params)
print("Best CV accuracy:", study_tfidf.best_value)

[I 2025-03-13 19:35:48,596] A new study created in memory with name: no-name-6faeacfc-8406-402d-b731-37ed9e67c409
[I 2025-03-13 19:36:13,183] Trial 0 finished with value: 0.7487748904111872 and parameters: {'max_df': 0.7567163524333893, 'min_df': 0.006687472628290311, 'ngram_high': 2, 'C': 0.3216424702619692, 'solver': 'liblinear', 'penalty': 'l1'}. Best is trial 0 with value: 0.7487748904111872.
[I 2025-03-13 19:36:34,023] Trial 1 finished with value: 0.6798893127989732 and parameters: {'max_df': 0.8289352925235547, 'min_df': 0.00644767724195263, 'ngram_high': 2, 'C': 0.032180053941251306, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.7487748904111872.
[I 2025-03-13 19:37:03,036] Trial 2 finished with value: 0.7830427188026068 and parameters: {'max_df': 0.8745285415072241, 'min_df': 0.006411813450649182, 'ngram_high': 2, 'C': 1.9531235921192884, 'solver': 'lbfgs'}. Best is trial 2 with value: 0.7830427188026068.
[I 2025-03-13 19:37:24,374] Trial 3 finished with value: 0.6744891461

Best parameters for TfidfVectorizer pipeline:
{'max_df': 0.8984398019837769, 'min_df': 0.0035797169922928444, 'ngram_high': 1, 'C': 4.76352608541508, 'solver': 'liblinear', 'penalty': 'l2'}
Best CV accuracy: 0.7905596288147742


#### Финальная оценка на тестовой выборке.

Строим финальные модели с оптимальными параметрами и оцениваем их на отложенной выборке.
Для каждого пайплайна сначала выводим оценку на валидационной выборке, затем на тестовой.


In [20]:
# Модель для CountVectorizer
best_count_pipe = Pipeline([
    ("vec", CountVectorizer(
        max_df=study_count.best_params["max_df"],
        min_df=study_count.best_params["min_df"],
        ngram_range=(1, study_count.best_params["ngram_high"])
    )),
    ("clf", LogisticRegression(
        C=study_count.best_params["C"],
        solver=study_count.best_params["solver"],
        penalty=study_count.best_params["penalty"] if "penalty" in study_count.best_params else "l2",
        random_state=SEED,
        max_iter=1000
    ))
])

In [21]:
best_count_pipe.fit(X_train, y_train)
val_preds_count = best_count_pipe.predict(X_valid)
test_preds_count = best_count_pipe.predict(X_test)
print("Final Report on Validation Set (CountVectorizer):")
print(classification_report(y_valid, val_preds_count))

Final Report on Validation Set (CountVectorizer):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       1.00      0.19      0.33        36
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.58      0.32      0.41       193
      Бывший СССР       0.83      0.76      0.79      1420
              Дом       0.85      0.74      0.79       578
         Из жизни       0.62      0.55      0.58       743
   Интернет и СМИ       0.75      0.69      0.72      1236
             Крым       1.00      0.06      0.12        16
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.86      0.86      0.86      1468
          Легпром       0.00      0.00      0.00         2
              Мир       0.77      0.83      0.80      3699
  Наука и техника       0.80      0.81      0.80      1426
      Путешествия       0.81      0.56      0.66       171
     

In [22]:
print("Final Report on Test Set (CountVectorizer):")
print(classification_report(y_test, test_preds_count))

Final Report on Test Set (CountVectorizer):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         4
   69-я параллель       0.60      0.09      0.15        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.55      0.28      0.37       194
      Бывший СССР       0.83      0.77      0.79      1420
              Дом       0.84      0.79      0.82       578
         Из жизни       0.63      0.57      0.60       744
   Интернет и СМИ       0.72      0.67      0.69      1236
             Крым       1.00      0.24      0.38        17
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.85      0.86      0.85      1467
          Легпром       0.00      0.00      0.00         2
              Мир       0.77      0.82      0.79      3699
  Наука и техника       0.80      0.83      0.81      1426
      Путешествия       0.73      0.55      0.63       171
           

In [23]:
# Модель для TfidfVectorizer
best_tfidf_pipe = Pipeline([
    ("vec", TfidfVectorizer(
        max_df=study_tfidf.best_params["max_df"],
        min_df=study_tfidf.best_params["min_df"],
        ngram_range=(1, study_tfidf.best_params["ngram_high"])
    )),
    ("clf", LogisticRegression(
        C=study_tfidf.best_params["C"],
        solver=study_tfidf.best_params["solver"],
        penalty=study_tfidf.best_params["penalty"] if "penalty" in study_tfidf.best_params else "l2",
        random_state=SEED,
        max_iter=1000
    ))
])

In [24]:
best_tfidf_pipe.fit(X_train, y_train)
val_preds_tfidf = best_tfidf_pipe.predict(X_valid)
test_preds_tfidf = best_tfidf_pipe.predict(X_test)
print("Final Report on Validation Set (TfidfVectorizer):")
print(classification_report(y_valid, val_preds_tfidf))

Final Report on Validation Set (TfidfVectorizer):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       1.00      0.17      0.29        36
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.68      0.22      0.34       193
      Бывший СССР       0.81      0.78      0.80      1420
              Дом       0.85      0.75      0.80       578
         Из жизни       0.63      0.50      0.56       743
   Интернет и СМИ       0.74      0.69      0.72      1236
             Крым       1.00      0.12      0.22        16
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.86      0.87      0.86      1468
          Легпром       0.00      0.00      0.00         2
              Мир       0.77      0.83      0.80      3699
  Наука и техника       0.80      0.81      0.80      1426
      Путешествия       0.82      0.61      0.70       171
     

In [25]:
print("Final Report on Test Set (TfidfVectorizer):")
print(classification_report(y_test, test_preds_tfidf))

Final Report on Test Set (TfidfVectorizer):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         4
   69-я параллель       0.75      0.09      0.15        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.62      0.22      0.33       194
      Бывший СССР       0.81      0.79      0.80      1420
              Дом       0.85      0.79      0.82       578
         Из жизни       0.65      0.54      0.59       744
   Интернет и СМИ       0.75      0.68      0.71      1236
             Крым       1.00      0.18      0.30        17
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.85      0.87      0.86      1467
          Легпром       0.00      0.00      0.00         2
              Мир       0.77      0.82      0.80      3699
  Наука и техника       0.81      0.84      0.82      1426
      Путешествия       0.78      0.63      0.69       171
           

Перезапустил на семпле из 100к.
Чуть лучше отработал TfidfVectorizer - {'max_df': 0.8984398019837769, 'min_df': 0.0035797169922928444, 'ngram_high': 1, 'C': 4.76352608541508, 'solver': 'liblinear', 'penalty': 'l2'}
- Best CV accuracy: 0.7905596288147742
- validation: accuracy 0.80
- test: accuracy 0.80